# Read the Target Sequence / Profile

In [1]:
import pandas as pd

family_df = pd.read_csv('protein_families.csv')

i = 0
name, accession, match_len, seed_len, type_, clan = family_df.iloc[i]
# match_len = 50

# Parse the sequences

In [2]:
from Bio import SeqIO
from modules.sequences import split_train_val_test

# read the fasta files
recs = [(r.id, str(r.seq)) for r in SeqIO.parse(f"sequences/{accession}.fasta", "fasta")]

# separate to train, validation, and test set
train_seqs, val_seqs, test_seqs = split_train_val_test(recs)

# Initialize HMM

In [3]:
from modules.model import (
    init_transition_mat,
    remove_silent_states,
    init_emission_mat
)

# init transition and emmission matrices
trans_matdf = init_transition_mat(match_len)
emit_matdf = init_emission_mat(match_len)

# remove silent D states
cols = list(trans_matdf.columns)
d_cols = [i for i in cols if i[0] == 'D']
trans_mat, red_cols = remove_silent_states(trans_matdf.to_numpy(), cols, d_cols)
trans_matdf = pd.DataFrame(trans_mat, columns=red_cols, index=red_cols)

# encode the symbols into index for the hmmlearn
symbols = list(emit_matdf.index)            # ^ A C D ... Y $  (taken from the CSV row order)
sym_idx = {s: i for i, s in enumerate(symbols)}
BEGIN = sym_idx["^"]                         # dummy begin symbol, emitted only by M0
END = sym_idx["$"]    

# Construct the Kraus Operators

In [4]:
from modules.stochastic_to_Kraus import kraus_operators, observable_T

A = trans_mat
B = emit_matdf.to_numpy()

K = kraus_operators(A, B)

# Forward Pass

In [5]:
import numpy as np
import torch

In [6]:
K_torch = {
    key: torch.tensor(
        value,
        dtype=torch.complex128,
        requires_grad=True
    )
    for key, value in K.items()
}

In [9]:
def dagger(A):
    return A.conj().T

# forward pass
def loss_function(p, K, W, seq):
    for s in seq:
        p_copy = torch.zeros_like(p)
        for w in range(W):
            p_copy = (
                p_copy
                + K[w, s] @ p @ dagger(K[w, s])
            )
    
        p = p_copy
    return (-torch.log(p.trace())).real

def stack_matrices(A):
    keys = sorted(A.keys(), key=lambda k: (k[1], k[0]))
    return torch.cat([A[key] for key in keys], dim=0)

def construct_UV(G_stacked, kappa):
    U = torch.cat([G_stacked, kappa], dim=1)
    V = torch.cat([kappa, -G_stacked], dim=1)
    return U, V

def kappa_tau(tau, kappa, U, V):
    I = torch.eye(
        U.shape[1],
        dtype=U.dtype,
        device=U.device
    )

    A = I + (tau / 2) * (dagger(V) @ U)
    B = dagger(V) @ kappa

    new_kappa = kappa - tau * U @ torch.linalg.solve(A, B)

    return new_kappa

def get_dict_from_stacked(stacked_K, keys):
    keys = sorted(keys, key=lambda k: (k[1], k[0]))
    n = stacked_K.shape[1]

    # if stacked_K.shape[0] != len(keys) * n:
    #     raise ValueError("Stacked matrix dimensions do not match K_torch.")

    return {
        key: stacked_K[i*n:(i+1)*n, :]
        for i, key in enumerate(keys)
    }

In [10]:
from modules.sequences import encode_sequences

p = torch.zeros(A.shape, dtype=torch.complex128)
p[0][0] = 1.0
seqs, len_ = encode_sequences(sym_idx, train_seqs)
W = 79

L = loss_function(p, K_torch, W, seqs[1:len_[0]].reshape(-1))
print(L)
# p.trace()

L.backward()

tensor(152.8049, dtype=torch.float64, grad_fn=<SelectBackward0>)


In [23]:
seqs[1:len_[0]].reshape(-1)

array([11, 13, 15,  2, 17,  7,  2,  3,  6,  7, 18, 16,  3,  3,  5, 18, 15,
       18,  5,  1,  1,  4,  3,  6, 14, 18, 15,  1,  2, 13, 12,  2,  1,  1,
       12,  1,  6,  8,  1,  4, 18, 16, 15,  3, 15,  6, 15,  9,  1, 21])

In [36]:
from modules.qevaluation import description_accuracy_hqmm
import time

# Initialization
p = torch.zeros(A.shape, dtype=torch.complex128)
p[0, 0] = 1.0

seqs, len_ = encode_sequences(sym_idx, train_seqs)
train_encoded = [
    encode_sequences(sym_idx, [seq])[0].reshape(-1)[1:]
    for seq in train_seqs
]
train_encoded = [
    seq for seq in train_encoded if len(seq) > 0
]

W = 79
epochs = 100
tau = 0.001

keys = sorted(K_torch.keys(), key=lambda k: (k[1], k[0]))
losses = []

history, cpu = [], 0.0
verbose=True

# COSM training loop
for epoch in range(epochs):

    # 1. Prepare Kraus operators for differentiation
    K_torch = {
        key: K_torch[key].detach().clone().requires_grad_(True)
        for key in keys
    }

    t0 = time.process_time()

    # 2. Compute loss
    L = torch.sum(
        loss_function(p, K_torch, W, seq)
        for seq in train_encoded
    )
    print('done sum L')

    # 3. Compute gradients
    L.backward()

    # 4. Extract Wirtinger gradients
    G = {
        key: (
            K_torch[key].grad / 2
            if K_torch[key].grad is not None
            else torch.zeros_like(K_torch[key])
        )
        for key in keys
    }

    # 5. Stack Kraus operators and gradients
    kappa = stack_matrices(K_torch).detach()
    G_stacked = stack_matrices(G).detach()

    # 6. Construct U and V
    U, V = construct_UV(G_stacked, kappa)

    # 7. Wen-Yin retraction
    with torch.no_grad():
        new_kappa = kappa_tau(tau, kappa, U, V)

    # 8. Convert stacked Kraus operators back to dictionary
    K_torch = get_dict_from_stacked(new_kappa, keys)

    cpu += time.process_time() - t0

    # 9. Record loss
    # losses.append(L.item())

    # 10. Get the validation DA and test DA
    with torch.no_grad():
        da = description_accuracy_hqmm(p, K_torch, W, sym_idx, val_seqs)
        for name, seqs in (("val", val_seqs), ("test", test_seqs)):
            if seqs:
                row = {"iter": epoch, "cpu_time": cpu, "loglik": L.item()}
                row[f"{name}_da_mean"], row[f"{name}_da_std"] = da.mean(), da.std()
        history.append(row)
        if verbose:
            extra = "".join(f"  {k} {v:.4f}" for k, v in row.items() if k.endswith("da_mean"))
            print(f"iter {epoch:3d}  cpu {cpu:7.1f}s  log-lik {L.item():.2f}{extra}")

TypeError: sum(): argument 'input' (position 1) must be Tensor, not generator

In [9]:
train_encoded = [
    encode_sequences(sym_idx, [seq])[0].reshape(-1)[1:]
    for seq in train_seqs
]

In [ ]:
for seq in train_encoded:
    print(loss_function(p, K_torch, W, seq))
    # break

tensor(152.1376, dtype=torch.float64, grad_fn=<SelectBackward0>)
tensor(245.3567, dtype=torch.float64, grad_fn=<SelectBackward0>)
tensor(274.2176, dtype=torch.float64, grad_fn=<SelectBackward0>)
tensor(156.6889, dtype=torch.float64, grad_fn=<SelectBackward0>)
tensor(140.4232, dtype=torch.float64, grad_fn=<SelectBackward0>)
tensor(209.0500, dtype=torch.float64, grad_fn=<SelectBackward0>)
tensor(145.7359, dtype=torch.float64, grad_fn=<SelectBackward0>)
tensor(154.2392, dtype=torch.float64, grad_fn=<SelectBackward0>)
tensor(234.3185, dtype=torch.float64, grad_fn=<SelectBackward0>)
tensor(201.4969, dtype=torch.float64, grad_fn=<SelectBackward0>)
tensor(213.2919, dtype=torch.float64, grad_fn=<SelectBackward0>)
tensor(151.9798, dtype=torch.float64, grad_fn=<SelectBackward0>)
tensor(234.5017, dtype=torch.float64, grad_fn=<SelectBackward0>)
tensor(187.4963, dtype=torch.float64, grad_fn=<SelectBackward0>)
tensor(197.5507, dtype=torch.float64, grad_fn=<SelectBackward0>)
tensor(474.9333, dtype=to

In [ ]:
[loss_function(p, K_torch, W, seq) for seq in train_encoded]

In [18]:
from modules.evaluation import (
    plot_da_vs_time,
    plot_da_bars,
)
import matplotlib.pyplot as plt

try:
    plt.style.use("seaborn-v0_8-darkgrid")   # close to the paper's look
except OSError:
    pass

for label, h in histories.items():
    h.to_csv(f"history_{label.split()[0]}.csv", index=False)
pd.DataFrame(results, index=["test_da_mean", "test_da_std"]).T.to_csv("test_da_results.csv")

# references = {}

AttributeError: 'list' object has no attribute 'items'

In [20]:
pd.DataFrame(history)[['iter', '']]

,iter,cpu_time,loglik,DUF7563_da_mean,DUF7563_da_std
0,0,12.610563,144.446734,0.004366,0.006210
1,1,28.812224,142.667005,0.006375,0.008158
2,2,42.438128,140.941135,0.008594,0.010125
3,3,55.165098,139.274354,0.010890,0.012162
4,4,67.796878,137.669063,0.013138,0.014271
5,5,81.884960,136.125223,0.015284,0.016420
6,6,93.185726,134.641244,0.017318,0.018572
7,7,105.341557,133.214672,0.019336,0.020616
8,8,118.079335,131.842632,0.021253,0.022625
9,9,131.958958,130.522102,0.023060,0.024600
